In [9]:
from openai import OpenAI
import os
import smtplib
import json
from email.mime.text import MIMEText
from datetime import datetime
import logging
from tools import weather_inquir,send_email
import ast

##### function call 用例
agent: openAI  |  模型: deepseek v4 pro | 运行环境：vscode

In [10]:
# 日志
logging.basicConfig(
    level=logging.DEBUG,  # 设置日志级别
    format='%(asctime)s - %(levelname)s - %(message)s',  # 日志格式
    filename='function call.log',  # 日志文件名（如果需要将日志保存到文件）
    filemode='a',  # 文件模式：'a' 表示追加日志，'w' 表示覆盖日志
    encoding="UTF-8"
)

# 获取日志记录器
logger = logging.getLogger()

In [11]:
# 工具接口描述，方便AI标准化地调用工具
tools_list = [
    {
        'type':'function',
        'function':{
            'name':'weather_inquir',
            'description':'get weather of a location ',#'get weather of a location in specific time',
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "a location or a city",
                    },
                    # "time":{
                    #     "type":"string",
                    #     "description":"date format 'YYYY-MM-DD', default now",
                    # },
                },
                # "required": ["city",'time'],
                "required": ["city"],
            },
        },
    },
    {
        'type':'function',
        'function':{
            'name':'send_email',
            'description':'send a message with nessecery text, no sttachment',
            'parameters':{
                "type": "object",
                "properties": {
                    "from_email":{
                        "type":"string",
                        "description":"a email address which send the email",
                    },
                    "to_email":{
                        "type":"string",
                        "description":"a email address which get the email",
                    },
                    "subject":{
                        "type":"string",
                        "description":"email subject text",
                    },
                    "body":{
                        "type":"string",
                        "description":"email body content text",
                    },
                },
                'required':['from_email','to_email','subject','body'],
            },
        },
    },
]

In [12]:
# AI配置
with open('config.json','r',encoding='utf-8') as file:
    config = json.load(file)

model_ongoing = config.get('model_ongoing') # 当前模型

client = OpenAI(
    api_key = config.get('api_key'), # api key
    base_url = config.get('model_url') # api url
    )

In [ ]:
# 测试用例，用户发送的信息
user_msg = config.get('user_msg') 

['hello',
 '查询今天上海天气',
 '帮我发一封邮件',
 '从haihua.yao@foxmail.com发送给1187522455@qq.com，内容随意，标题是测试',
 '再见']

In [14]:
# 对话流
msg_stream = [{"role": "system", "content": "You are a helpful assistant"}]
# msg_stream = []
# ai返回结果
df = []
tool_index = 0
for i in range(len(user_msg)):
    current_tool_calls = []
    # 更新对话流
    msg_stream.append({'role':'user','content':user_msg[i]})

    print(msg_stream)
    response = client.chat.completions.create(
        model=model_ongoing,
        messages=msg_stream,
        tools= tools_list,
        stream=False,
        reasoning_effort="high", # 'max
        extra_body={"thinking": {"type": "enabled"}} # 'disabled'
    )

    # 找到这次需要调用的工具
    compete_tool_calls = response.choices[0].message.tool_calls

    if compete_tool_calls:
        if len(compete_tool_calls) > tool_index: # 有新的tool调用
            current_tool_calls = compete_tool_calls[tool_index:]
            tool_index = len(compete_tool_calls)

    # 本次AI补全结果
    result = {
        'massage':response.choices[0].message.content,
        'function_call':response.choices[0].message.function_call,
        'tool_call':current_tool_calls, # response.choices[0].message.tool_calls,
        'finish_reason':response.choices[0].finish_reason,
        'token':response.usage.total_tokens
    }
    df.append(result)

    # logging
    logger.info(f"role = system massage = {result['massage']} function_call = {result['function_call']} tool_call = {result['tool_call']} ") 

    # 处理工具调用
    # print(result['tool_call']) # 当前工具列表
    
    if result['tool_call']:
        print('调用工具')
        for tool in result['tool_call']:
            # TODO
            tool_name = tool.function.name
            tool_args = tool.function.arguments
            print(f"调用{tool_name}")
            # 调用
            args = ast.literal_eval(tool_args)
            if tool_name == 'weather_inquir':
                tool_result = weather_inquir(args['city'])
            elif tool_name == 'send_email':
                tool_result = send_email(args['from_email'],args['to_email'],args['subject'],args['body'])
            else:
                tool_result = {"error":f"unknown tool+{tool_name}"}

            #更新对话流
            if tool_result:
                msg_stream.append({"role":"system","content":tool_result})
                print("工具调用结果更新完毕")
            else:
                msg_stream.append({"role":"system","content":'找不到工具，或没有结果'})
                print("工具调用没有结果")
    else:
        # 更新对话流
        msg_stream.append({"role":"system","content":result['massage']})
        print("AI补全对话")
        logger.info(f"role = system massage = {result['massage']} function_call = {result['function_call']} tool_call = {result['tool_call']} ") 



[{'role': 'system', 'content': 'You are a helpful assistant'}, {'role': 'user', 'content': 'hello'}]
AI补全对话
[{'role': 'system', 'content': 'You are a helpful assistant'}, {'role': 'user', 'content': 'hello'}, {'role': 'system', 'content': 'Hello! 😊 How can I help you today?'}, {'role': 'user', 'content': '查询今天上海天气'}]
调用工具
调用weather_inquir
查询完毕，上海天气是：小雨,27℃
工具调用结果更新完毕
[{'role': 'system', 'content': 'You are a helpful assistant'}, {'role': 'user', 'content': 'hello'}, {'role': 'system', 'content': 'Hello! 😊 How can I help you today?'}, {'role': 'user', 'content': '查询今天上海天气'}, {'role': 'system', 'content': '查询完毕，上海天气是：小雨,27℃'}, {'role': 'user', 'content': '帮我发一封邮件'}]
AI补全对话
[{'role': 'system', 'content': 'You are a helpful assistant'}, {'role': 'user', 'content': 'hello'}, {'role': 'system', 'content': 'Hello! 😊 How can I help you today?'}, {'role': 'user', 'content': '查询今天上海天气'}, {'role': 'system', 'content': '查询完毕，上海天气是：小雨,27℃'}, {'role': 'user', 'content': '帮我发一封邮件'}, {'role': 'system'

In [15]:
msg_stream

[{'role': 'system', 'content': 'You are a helpful assistant'},
 {'role': 'user', 'content': 'hello'},
 {'role': 'system', 'content': 'Hello! 😊 How can I help you today?'},
 {'role': 'user', 'content': '查询今天上海天气'},
 {'role': 'system', 'content': '查询完毕，上海天气是：小雨,27℃'},
 {'role': 'user', 'content': '帮我发一封邮件'},
 {'role': 'system', 'content': '你好！我来帮你查询上海天气，不过关于邮件我还需要一些信息。先查天气：'},
 {'role': 'user',
  'content': '从haihua.yao@foxmail.com发送给1187522455@qq.com，内容随意，标题是测试'},
 {'role': 'system', 'content': '邮件发送成功，还有什么需要帮助您的吗？'},
 {'role': 'user', 'content': '再见'},
 {'role': 'system', 'content': '你好！我来同时帮你查询上海天气和发送邮件。'}]